# SQL Window Functions — Runnable Examples

This notebook uses Python's built-in `sqlite3` database. SQLite versions with window-function support (3.25+) can run these examples. Each example creates or queries a small in-memory sales table.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("""CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    salesperson TEXT,
    region TEXT,
    sale_date TEXT,
    amount REAL
)""")
rows = [
    (1, "Asha", "North", "2026-01-03", 100),
    (2, "Ravi", "North", "2026-01-04", 150),
    (3, "Asha", "North", "2026-01-05", 200),
    (4, "Neha", "South", "2026-01-03", 120),
    (5, "Ravi", "South", "2026-01-06", 180),
    (6, "Neha", "South", "2026-01-07", 180),
]
conn.executemany("INSERT INTO sales VALUES (?, ?, ?, ?, ?)", rows)
pd.read_sql_query("SELECT * FROM sales ORDER BY sale_date, sale_id", conn)


,sale_id,salesperson,region,sale_date,amount
0,1,Asha,North,2026-01-03,100.0
1,4,Neha,South,2026-01-03,120.0
2,2,Ravi,North,2026-01-04,150.0
3,3,Asha,North,2026-01-05,200.0
4,5,Ravi,South,2026-01-06,180.0
5,6,Neha,South,2026-01-07,180.0


## 1. ROW_NUMBER, RANK, and DENSE_RANK

Use ranking functions to order sales within each region.

In [2]:
pd.read_sql_query("""
SELECT sale_id, salesperson, region, amount,
       ROW_NUMBER() OVER (
           PARTITION BY region ORDER BY amount DESC, sale_id
       ) AS row_num,
       RANK() OVER (
           PARTITION BY region ORDER BY amount DESC
       ) AS sales_rank,
       DENSE_RANK() OVER (
           PARTITION BY region ORDER BY amount DESC
       ) AS dense_rank
FROM sales
ORDER BY region, amount DESC, sale_id
""", conn)


,sale_id,salesperson,region,amount,row_num,sales_rank,dense_rank
0,3,Asha,North,200.0,1,1,1
1,2,Ravi,North,150.0,2,2,2
2,1,Asha,North,100.0,3,3,3
3,5,Ravi,South,180.0,1,1,1
4,6,Neha,South,180.0,2,1,1
5,4,Neha,South,120.0,3,3,2


## 2. Region totals without collapsing rows

`SUM() OVER (PARTITION BY region)` retains each individual sale.

In [3]:
pd.read_sql_query("""
SELECT sale_id, region, amount,
       SUM(amount) OVER (PARTITION BY region) AS region_total
FROM sales
ORDER BY region, sale_id
""", conn)


,sale_id,region,amount,region_total
0,1,North,100.0,450.0
1,2,North,150.0,450.0
2,3,North,200.0,450.0
3,4,South,120.0,480.0
4,5,South,180.0,480.0
5,6,South,180.0,480.0


## 3. Running total

An explicit `ROWS` frame plus a unique order makes the cumulative calculation clear.

In [4]:
pd.read_sql_query("""
SELECT sale_id, sale_date, amount,
       SUM(amount) OVER (
           ORDER BY sale_date, sale_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM sales
ORDER BY sale_date, sale_id
""", conn)


,sale_id,sale_date,amount,running_total
0,1,2026-01-03,100.0,100.0
1,4,2026-01-03,120.0,220.0
2,2,2026-01-04,150.0,370.0
3,3,2026-01-05,200.0,570.0
4,5,2026-01-06,180.0,750.0
5,6,2026-01-07,180.0,930.0


## 4. Three-row moving average

In [5]:
pd.read_sql_query("""
SELECT sale_id, sale_date, amount,
       ROUND(AVG(amount) OVER (
           ORDER BY sale_date, sale_id
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
       ), 2) AS moving_avg_3_rows
FROM sales
ORDER BY sale_date, sale_id
""", conn)


,sale_id,sale_date,amount,moving_avg_3_rows
0,1,2026-01-03,100.0,100.00
1,4,2026-01-03,120.0,110.00
2,2,2026-01-04,150.0,123.33
3,3,2026-01-05,200.0,156.67
4,5,2026-01-06,180.0,176.67
5,6,2026-01-07,180.0,186.67


## 5. LAG and LEAD

These functions compare a row with its predecessor and successor.

In [6]:
pd.read_sql_query("""
SELECT sale_id, sale_date, amount,
       LAG(amount) OVER (ORDER BY sale_date, sale_id) AS previous_amount,
       LEAD(amount) OVER (ORDER BY sale_date, sale_id) AS next_amount,
       amount - LAG(amount) OVER (ORDER BY sale_date, sale_id) AS change_from_previous
FROM sales
ORDER BY sale_date, sale_id
""", conn)


,sale_id,sale_date,amount,previous_amount,next_amount,change_from_previous
0,1,2026-01-03,100.0,NaN,120.0,NaN
1,4,2026-01-03,120.0,100.0,150.0,20.0
2,2,2026-01-04,150.0,120.0,200.0,30.0
3,3,2026-01-05,200.0,150.0,180.0,50.0
4,5,2026-01-06,180.0,200.0,180.0,-20.0
5,6,2026-01-07,180.0,180.0,NaN,0.0


## 6. First and last values in a partition

In [7]:
pd.read_sql_query("""
SELECT sale_id, salesperson, sale_date, amount,
       FIRST_VALUE(amount) OVER (
           PARTITION BY salesperson ORDER BY sale_date, sale_id
       ) AS first_amount,
       LAST_VALUE(amount) OVER (
           PARTITION BY salesperson ORDER BY sale_date, sale_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
       ) AS last_amount
FROM sales
ORDER BY salesperson, sale_date, sale_id
""", conn)


,sale_id,salesperson,sale_date,amount,first_amount,last_amount
0,1,Asha,2026-01-03,100.0,100.0,200.0
1,3,Asha,2026-01-05,200.0,100.0,200.0
2,4,Neha,2026-01-03,120.0,120.0,180.0
3,6,Neha,2026-01-07,180.0,120.0,180.0
4,2,Ravi,2026-01-04,150.0,150.0,180.0
5,5,Ravi,2026-01-06,180.0,150.0,180.0


## 7. Percent of total and NTILE

In [8]:
pd.read_sql_query("""
SELECT sale_id, amount,
       ROUND(100.0 * amount / SUM(amount) OVER (), 2) AS pct_of_total,
       NTILE(4) OVER (ORDER BY amount, sale_id) AS amount_bucket
FROM sales
ORDER BY amount, sale_id
""", conn)


,sale_id,amount,pct_of_total,amount_bucket
0,1,100.0,10.75,1
1,4,120.0,12.90,1
2,2,150.0,16.13,2
3,5,180.0,19.35,2
4,6,180.0,19.35,3
5,3,200.0,21.51,4


## 8. Top two sales per region

Use a CTE to filter after calculating the window rank.

In [9]:
pd.read_sql_query("""
WITH ranked AS (
    SELECT sale_id, salesperson, region, amount,
           ROW_NUMBER() OVER (
               PARTITION BY region ORDER BY amount DESC, sale_id
           ) AS rn
    FROM sales
)
SELECT sale_id, salesperson, region, amount
FROM ranked
WHERE rn <= 2
ORDER BY region, amount DESC, sale_id
""", conn)


,sale_id,salesperson,region,amount
0,3,Asha,North,200.0
1,2,Ravi,North,150.0
2,5,Ravi,South,180.0
3,6,Neha,South,180.0


## 9. Compare each sale to its region average

In [10]:
pd.read_sql_query("""
SELECT sale_id, salesperson, region, amount,
       ROUND(AVG(amount) OVER (PARTITION BY region), 2) AS region_avg,
       ROUND(amount - AVG(amount) OVER (PARTITION BY region), 2) AS difference_from_avg
FROM sales
ORDER BY region, sale_id
""", conn)


,sale_id,salesperson,region,amount,region_avg,difference_from_avg
0,1,Asha,North,100.0,150.0,-50.0
1,2,Ravi,North,150.0,150.0,0.0
2,3,Asha,North,200.0,150.0,50.0
3,4,Neha,South,120.0,160.0,-40.0
4,5,Ravi,South,180.0,160.0,20.0
5,6,Neha,South,180.0,160.0,20.0


## 10. Cumulative count and cleanup

In [11]:
pd.read_sql_query("""
SELECT sale_id, sale_date,
       COUNT(*) OVER (
           ORDER BY sale_date, sale_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS cumulative_sales_count
FROM sales
ORDER BY sale_date, sale_id
""", conn)


,sale_id,sale_date,cumulative_sales_count
0,1,2026-01-03,1
1,4,2026-01-03,2
2,2,2026-01-04,3
3,3,2026-01-05,4
4,5,2026-01-06,5
5,6,2026-01-07,6


## Key takeaways

- Window functions preserve row-level detail.
- `PARTITION BY` controls the groups; `ORDER BY` controls the analytic order.
- Explicit frames make running totals and moving calculations predictable.
- Use a CTE or subquery to filter calculated ranks.